# CardI-HACK Challenge  V45 - ADVANCED

==================================


Combine multiple MACE models like we did for severity.
Average their probabilities, then apply thresholds.


#  Requirements
Dependencies

**Python 3.8+**

**Core Data Science
numpy>=1.21.0
pandas>=1.3.0
scipy>=1.7.0

**Machine Learning
scikit-learn>=1.0.0

**Optional: For faster computation
 joblib>=1.1.0  # Already included in scikit-learn**


#  Methodology

### CardioHack Methodology

### Final Submission: `phase9_ensemble_weighted.csv` | Score: 0.3240

---

### 1. Problem Overview

**Objective**: Predict two cardiovascular outcomes:
- **MACE** (3-class): Major Adverse Cardiac Events (0, 1, 2)
- **Severity** (probability): Event severity [0, 1]

**Scoring**:
```
Score = 0.7 × QWK(MACE) + 0.3 × Rescaled_WLL(Severity)
```

**Dataset**: 347 training samples, 149 test samples, 288 SNP features

---

### 2. Approach Selection Rationale

We experimented with 20+ approaches before selecting the final submission:

| Approach | Score | Decision |
|----------|-------|----------|
| **6-model ensemble (Phase 9)** | **0.3240** | ✅ Selected |
| Simple LR baseline  |  Good baseline same version as yesterday |
| Ordinal regression |  ❌ Worse metric not being tested|
| Tree-based + SMOTE |  ❌ Overfit Class 0 and Worse metric not being tested|
| Advanced SNP features | ❌ Added noise and Worse metric not being tested|
| Target encoding | ❌ Data leakage enven with better metric not being tested |

**Key Learning**: Simple models with careful ensemble weighting outperform complex approaches on small datasets (n=347).

---

## 3. Feature Engineering

### Clinical Features
```python
Age_bin = pd.cut(Age_Baseline, bins=[0, 50, 60, 70, 100], labels=[0, 1, 2, 3])
Age_sq = Age_Baseline ** 2
Genre = Genre.map({'M': 1, 'F': 0}).fillna(0.5)
```

### SNP Selection Pipeline
1. **MAF Filter**: Remove SNPs with minor allele frequency < 0.5%
2. **Chi-squared Test**: Keep SNPs with p-value < 0.10 (MACE) or < 0.15 (Severity)
3. **Priority Enforcement**: At least 20% from SNP1-75 (domain priority)
4. **Final Count**: ~60 features for MACE

---

### 4. MACE Model

### Architecture: 6-Model Ensemble

| Model | Type | C | l1_ratio | Other |
|-------|------|---|----------|-------|
| LR1 | LogisticRegression | 0.3 | 0.3 | balanced |
| LR2 | LogisticRegression | 0.5 | 0.3 | balanced |
| LR3 | LogisticRegression | 0.3 | 0.5 | balanced |
| GB | GradientBoosting | - | - | n=100, d=3 |
| RF | RandomForest | - | - | n=200, d=5 |
| LGB | LightGBM | - | - | n=100, d=4 |

### Ensemble Strategy

**Step 1**: Out-of-fold predictions
```python
cv = StratifiedKFold(5, shuffle=True, random_state=42)
oof_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')
```

**Step 2**: QWK-based weighting
```python
# Apply thresholds to get class predictions
pred = apply_thresholds(oof_proba, t1=0.35, t2=0.50)
qwk = cohen_kappa_score(y, pred, weights='quadratic')

# Exponential weighting emphasizes best models
weight = exp(qwk * 5) / sum(exp(all_qwk * 5))
```

**Step 3**: Weighted probability averaging
```python
ensemble_proba = sum(model_proba[k] * weights[k] for k in models)
```

**Step 4**: Apply thresholds for final prediction
```python
pred = zeros(n)
pred[proba[:, 2] > 0.50] = 2  # Class 2 if P(2) > 0.50
pred[(proba[:, 1] > 0.35) & (proba[:, 2] <= 0.50)] = 1  # Class 1 if P(1) > 0.35
```

### Model Weights (from OOF QWK)

| Model | QWK | Weight |
|-------|-----|--------|
| lr1 | 0.298 | 0.258 |
| lr2 | 0.266 | 0.220 |
| lr3 | 0.241 | 0.194 |
| lgb | 0.213 | 0.169 |
| gb | 0.110 | 0.100 |
| rf | 0.007 | 0.060 |

**Insight**: LR models dominate the ensemble weight (67%), confirming that linear models work best for this problem.

---

### 5. Severity Model

### Architecture: 2-Model Ensemble

| Model | Type | Parameters |
|-------|------|------------|
| LR | LogisticRegression | C=0.5, l1_ratio=0.3, balanced |
| GB | GradientBoosting | n=50, d=2, lr=0.05 |

### Ensemble Strategy

**Brier-weighted averaging**:
```python
brier = brier_score_loss(y, oof_proba)
weight = (1/brier) / sum(1/all_brier)
```

**Isotonic calibration**:
```python
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_ensemble, y)
calibrated = iso.predict(test_ensemble)
final = clip(calibrated, 0.01, 0.99)
```

---

### 6. Why This Approach Won

| Factor | Explanation |
|--------|-------------|
| **Simple models** | LR generalizes better on 347 samples |
| **Ensemble weighting** | QWK-based weights emphasize best performers |
| **Fixed thresholds** | (0.35, 0.50) balanced sensitivity/specificity |
| **Calibration** | Isotonic regression improved severity |

---

### 7. Final Predictions

| Metric | Value |
|--------|-------|
| MACE Distribution | [118, 15, 16] (Class 0, 1, 2) |
| Severity Mean | 0.319 |
| **Final Score** | **0.3240** |

---

### 8. Reproducibility

```bash
# Environment
Python 3.9+

# Install
pip install -r requirements.txt

# Run
python cardihack_phase9_mace_ensemble.py

# Output
phase9_ensemble_weighted.csv
```


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import cohen_kappa_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGBM = True
except: HAS_LGBM = False

# ===================================================
# CONFIG
# ===================================================
ID_COL = "trustii_id"
AGE_COL = "Age_Baseline"
SEX_COL = "Genre"
TARGET_SEV = "OUTCOME SEVERITY"
TARGET_MACE = "OUTCOME MACE"
SNP_COLS = [f"SNP{i}" for i in range(1, 289)]
PRIORITY_SNP = [f"SNP{i}" for i in range(1, 76)]
DATA_DIR = "/content"

# ===================================================
# LOAD DATA
# ===================================================
print("=" * 60)
print("PHASE 9: MACE ENSEMBLE")
print("=" * 60)
train  = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
dummy = pd.read_csv(f"{DATA_DIR}/cardihack_dummy_pred-1762193921064.csv")

# Helpers
def filter_maf(df, snps, thresh=0.005):
    out = []
    for s in snps:
        v = df[s].dropna()
        if len(v) < 10: continue
        f = v.value_counts()
        if len(f) < 2: continue
        if f.min() / f.sum() >= thresh: out.append(s)
    return out

def chi2_snps(df, target, snps, p_thresh=0.1):
    pvals = []
    for s in snps:
        tab = pd.crosstab(df[s].fillna(-1), df[target])
        if tab.shape[0] < 2 or tab.shape[1] < 2: continue
        try:
            _, p, _, _ = chi2_contingency(tab)
            pvals.append((s, p))
        except: pass
    pvals.sort(key=lambda x: x[1])
    return [s for s, p in pvals if p < p_thresh]

def enforce_priority(selected, priority, max_total=120):
    pin = [s for s in selected if s in priority]
    other = [s for s in selected if s not in priority]
    minp = max(5, int(max_total * 0.2))
    if len(pin) < minp:
        add = [s for s in priority if s not in pin]
        pin.extend(add[:minp - len(pin)])
    return (pin + other)[:max_total]

def engineer(df):
    df = df.copy()
    if AGE_COL in df.columns:
        df['Age_bin'] = pd.cut(df[AGE_COL], bins=[0,50,60,70,100], labels=[0,1,2,3])
        df['Age_sq'] = df[AGE_COL] ** 2
    if SEX_COL in df.columns:
        df[SEX_COL] = df[SEX_COL].map({'M':1,'F':0}).fillna(0.5)
    return df

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

train = engineer(train)
test = engineer(test)

clinical = [c for c in train.columns
            if c not in SNP_COLS + [ID_COL, TARGET_MACE, TARGET_SEV]
            and c in test.columns and train[c].dtype in ['float64','int64']]

y_mace = train[TARGET_MACE].values
y_sev = train[TARGET_SEV].values

# Features
mace_snps = filter_maf(train, SNP_COLS, 0.005)
mace_snps = chi2_snps(train, TARGET_MACE, mace_snps, 0.10)
mace_snps = enforce_priority(mace_snps, PRIORITY_SNP, 120)

mace_feats = [AGE_COL, SEX_COL, 'Age_bin', 'Age_sq'] + clinical + mace_snps
mace_feats = [f for f in mace_feats if f in train.columns and f in test.columns]
mace_feats = list(dict.fromkeys(mace_feats))

imp = IterativeImputer(max_iter=20, random_state=42)
sc = RobustScaler()

X_train = sc.fit_transform(imp.fit_transform(train[mace_feats]))
X_test = sc.transform(imp.transform(test[mace_feats]))

print(f"\n📊 Features: {len(mace_feats)}")

cv = StratifiedKFold(5, shuffle=True, random_state=42)

# ===================================================
# TRAIN MULTIPLE MACE MODELS
# ===================================================
print("\n📊 TRAINING MACE ENSEMBLE")

mace_oof = {}
mace_test = {}

# Model 1: LR (V51 settings) - Our proven best
print("\n   Model 1: LR V51...")
lr1 = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.3,
                          max_iter=10000, multi_class="multinomial", class_weight='balanced', random_state=42)
lr1.fit(X_train, y_mace)
mace_oof['lr1'] = cross_val_predict(lr1, X_train, y_mace, cv=cv, method='predict_proba')
mace_test['lr1'] = lr1.predict_proba(X_test)

# Model 2: LR with different C
print("   Model 2: LR C=0.5...")
lr2 = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.5,
                          max_iter=10000, multi_class="multinomial", class_weight='balanced', random_state=42)
lr2.fit(X_train, y_mace)
mace_oof['lr2'] = cross_val_predict(lr2, X_train, y_mace, cv=cv, method='predict_proba')
mace_test['lr2'] = lr2.predict_proba(X_test)

# Model 3: LR with different l1_ratio
print("   Model 3: LR l1=0.5...")
lr3 = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=0.3,
                          max_iter=10000, multi_class="multinomial", class_weight='balanced', random_state=42)
lr3.fit(X_train, y_mace)
mace_oof['lr3'] = cross_val_predict(lr3, X_train, y_mace, cv=cv, method='predict_proba')
mace_test['lr3'] = lr3.predict_proba(X_test)

# Model 4: GradientBoosting
print("   Model 4: GradientBoosting...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                                 subsample=0.8, random_state=42)
gb.fit(X_train, y_mace)
mace_oof['gb'] = cross_val_predict(gb, X_train, y_mace, cv=cv, method='predict_proba')
mace_test['gb'] = gb.predict_proba(X_test)

# Model 5: RandomForest
print("   Model 5: RandomForest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_mace)
mace_oof['rf'] = cross_val_predict(rf, X_train, y_mace, cv=cv, method='predict_proba')
mace_test['rf'] = rf.predict_proba(X_test)

# Model 6: LightGBM
if HAS_LGBM:
    print("   Model 6: LightGBM...")
    lgb_m = lgb.LGBMClassifier(objective='multiclass', num_class=3, n_estimators=100,
                                max_depth=4, learning_rate=0.05, class_weight='balanced',
                                random_state=42, verbose=-1)
    lgb_m.fit(X_train, y_mace)
    mace_oof['lgb'] = cross_val_predict(lgb_m, X_train, y_mace, cv=cv, method='predict_proba')
    mace_test['lgb'] = lgb_m.predict_proba(X_test)

# ===================================================
# CALCULATE MODEL WEIGHTS (based on OOF QWK)
# ===================================================
print("\n📊 CALCULATING ENSEMBLE WEIGHTS")

t1, t2 = 0.35, 0.50  # V51 thresholds

model_qwk = {}
for name, oof in mace_oof.items():
    pred = np.zeros(len(y_mace), dtype=int)
    pred[oof[:, 2] > t2] = 2
    pred[(oof[:, 1] > t1) & (oof[:, 2] <= t2)] = 1
    model_qwk[name] = qwk(y_mace, pred)
    print(f"   {name}: QWK={model_qwk[name]:.4f}, Dist={np.bincount(pred, minlength=3)}")

# Weight by QWK (use exp to emphasize best models)
total = sum(np.exp(q*5) for q in model_qwk.values())
weights = {k: np.exp(v*5) / total for k, v in model_qwk.items()}

print("\n   Ensemble weights:")
for name, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"      {name}: {w:.3f}")

# ===================================================
# ENSEMBLE PREDICTIONS
# ===================================================
print("\n📊 ENSEMBLE PREDICTIONS")
results = {}

# Weighted average ensemble
oof_ensemble = sum(mace_oof[k] * weights[k] for k in weights)
test_ensemble = sum(mace_test[k] * weights[k] for k in weights)

# Apply thresholds
ens_pred = np.zeros(len(y_mace), dtype=int)
ens_pred[oof_ensemble[:, 2] > t2] = 2
ens_pred[(oof_ensemble[:, 1] > t1) & (oof_ensemble[:, 2] <= t2)] = 1
print(f"\n   Ensemble (weighted): QWK={qwk(y_mace, ens_pred):.4f}, Dist={np.bincount(ens_pred, minlength=3)}")

# Test predictions
test_ens_pred = np.zeros(len(X_test), dtype=int)
test_ens_pred[test_ensemble[:, 2] > t2] = 2
test_ens_pred[(test_ensemble[:, 1] > t1) & (test_ensemble[:, 2] <= t2)] = 1
results['ensemble_weighted'] = test_ens_pred

# Simple average ensemble
oof_simple = sum(mace_oof[k] for k in mace_oof) / len(mace_oof)
test_simple = sum(mace_test[k] for k in mace_test) / len(mace_test)

simple_pred = np.zeros(len(y_mace), dtype=int)
simple_pred[oof_simple[:, 2] > t2] = 2
simple_pred[(oof_simple[:, 1] > t1) & (oof_simple[:, 2] <= t2)] = 1
print(f"   Ensemble (simple avg): QWK={qwk(y_mace, simple_pred):.4f}, Dist={np.bincount(simple_pred, minlength=3)}")

test_simple_pred = np.zeros(len(X_test), dtype=int)
test_simple_pred[test_simple[:, 2] > t2] = 2
test_simple_pred[(test_simple[:, 1] > t1) & (test_simple[:, 2] <= t2)] = 1
results['ensemble_simple'] = test_simple_pred

# LR-only ensemble (top 3 LR models)
lr_oof = (mace_oof['lr1'] + mace_oof['lr2'] + mace_oof['lr3']) / 3
lr_test = (mace_test['lr1'] + mace_test['lr2'] + mace_test['lr3']) / 3

lr_pred = np.zeros(len(y_mace), dtype=int)
lr_pred[lr_oof[:, 2] > t2] = 2
lr_pred[(lr_oof[:, 1] > t1) & (lr_oof[:, 2] <= t2)] = 1
print(f"   Ensemble (LR only): QWK={qwk(y_mace, lr_pred):.4f}, Dist={np.bincount(lr_pred, minlength=3)}")

test_lr_pred = np.zeros(len(X_test), dtype=int)
test_lr_pred[lr_test[:, 2] > t2] = 2
test_lr_pred[(lr_test[:, 1] > t1) & (lr_test[:, 2] <= t2)] = 1
results['ensemble_lr_only'] = test_lr_pred

# ===================================================
# V51 SEVERITY
# ===================================================
print("\n📊 V51 Severity...")
sev_snps = filter_maf(train, SNP_COLS, 0.005)
sev_snps = chi2_snps(train, TARGET_SEV, sev_snps, 0.15)

sev_clinical = [AGE_COL, SEX_COL, 'Age_bin', 'Age_sq'] + clinical
sev_clinical = [f for f in sev_clinical if f in train.columns and f in test.columns]

imp_c = IterativeImputer(max_iter=20, random_state=42)
X_tr_c = imp_c.fit_transform(train[sev_clinical])
X_te_c = imp_c.transform(test[sev_clinical])

sev_oof, sev_test_preds = {}, {}

m1 = Pipeline([("sc", RobustScaler()), ("clf", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.5, max_iter=10000, class_weight='balanced', random_state=42))])
m1.fit(X_tr_c, y_sev); sev_oof['lr1'] = cross_val_predict(m1, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test_preds['lr1'] = m1.predict_proba(X_te_c)[:, 1]

m5 = GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, subsample=0.7, random_state=42)
m5.fit(X_tr_c, y_sev); sev_oof['gb2'] = cross_val_predict(m5, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test_preds['gb2'] = m5.predict_proba(X_te_c)[:, 1]

brier = {k: brier_score_loss(y_sev, v) for k, v in sev_oof.items()}
weights_sev = {k: (1/v) / sum(1/b for b in brier.values()) for k, v in brier.items()}

oof_ens_sev = sum(sev_oof[k] * weights_sev[k] for k in weights_sev)
test_ens_sev = sum(sev_test_preds[k] * weights_sev[k] for k in weights_sev)

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_ens_sev, y_sev)
sev_pred = np.clip(iso.predict(test_ens_sev), 0.01, 0.99)

# ===================================================
# SUBMISSIONS
# ===================================================
print("\n" + "=" * 60)
print("SUBMISSIONS")
print("=" * 60)

for name, mace_pred in results.items():
    filename = f"phase9_{name}.csv"
    sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET_MACE: mace_pred, TARGET_SEV: sev_pred})
    sub = sub[dummy.columns]
    sub.to_csv(f"{DATA_DIR}/{filename}", index=False)
    print(f"   {filename}: MACE {np.bincount(mace_pred, minlength=3)}")


PHASE 9: MACE ENSEMBLE

📊 Features: 60

📊 TRAINING MACE ENSEMBLE

   Model 1: LR V51...
   Model 2: LR C=0.5...
   Model 3: LR l1=0.5...
   Model 4: GradientBoosting...
   Model 5: RandomForest...
   Model 6: LightGBM...

📊 CALCULATING ENSEMBLE WEIGHTS
   lr1: QWK=0.2980, Dist=[209  63  75]
   lr2: QWK=0.2661, Dist=[198  75  74]
   lr3: QWK=0.2408, Dist=[179  86  82]
   gb: QWK=0.1096, Dist=[322  13  12]
   rf: QWK=0.0066, Dist=[321  25   1]
   lgb: QWK=0.2134, Dist=[271  39  37]

   Ensemble weights:
      lr1: 0.258
      lr2: 0.220
      lr3: 0.194
      lgb: 0.169
      gb: 0.100
      rf: 0.060

📊 ENSEMBLE PREDICTIONS

   Ensemble (weighted): QWK=0.3077, Dist=[234  61  52]
   Ensemble (simple avg): QWK=0.2281, Dist=[267  45  35]
   Ensemble (LR only): QWK=0.3155, Dist=[190  81  76]

📊 V51 Severity...

SUBMISSIONS
   phase9_ensemble_weighted.csv: MACE [118  15  16]
   phase9_ensemble_simple.csv: MACE [119  14  16]
   phase9_ensemble_lr_only.csv: MACE [105  20  24]


# README

### Chosen Submission: `phase9_ensemble_weighted.csv`

**Final Score: 0.3240** ✅

### Why This Submission?

After extensive experimentation with 20+ approaches, `phase9_ensemble_weighted.csv` achieved the highest leaderboard score. It uses a multi-model ensemble strategy that proved most robust on this dataset.

### Approach Summary

| Target | Method | Key Insight |
|--------|--------|-------------|
| **MACE** | 6-model ensemble + QWK-weighted averaging | Simple models generalize better |
| **Severity** | 2-model ensemble + isotonic calibration | Brier-weighted averaging works |

### Quick Start

```bash
pip install -r requirements.txt
python cardihack_phase9_mace_ensemble.py
```

### Output

- **File**: `phase9_ensemble_weighted.csv`
- **MACE Distribution**: [118, 15, 16]
- **Severity Mean**: 0.319

### Score Comparison

| Submission | Score | Improvement |
|------------|-------|-------------|
| **phase9_ensemble_weighted** | **0.3240** | +4.0% vs baseline |

### Team(author):
**earthy light**

### Files

| File | Description |
|------|-------------|
| `README.md` | This file |
| `METHODOLOGY.md` | Detailed technical methodology |
| `requirements.txt` | Python dependencies |
| `cardihack_phase9_mace_ensemble.py` | Main script |
